In [ ]:
import os
import shutil
import pydicom


source_path = 'D:\Datasets\AMCGH\AMCGH'

print(f"--- STARTING DATA CLEANING: {source_path} ---")

if os.path.exists(source_path):
    patient_folders = sorted([f for f in os.listdir(source_path) if os.path.isdir(os.path.join(source_path, f))])
    print(f"Found {len(patient_folders)} patients.")
    
    for i, patient_id in enumerate(patient_folders):
        patient_dir = os.path.join(source_path, patient_id)
        ct_dir = os.path.join(patient_dir, 'CT')
        
        if not os.path.exists(ct_dir):
            os.makedirs(ct_dir)
            
        print(f"[{i+1}] Organizing {patient_id}...", end='\r')
        
        files_moved = 0
        
       
        for f in os.listdir(patient_dir):
            full_path = os.path.join(patient_dir, f)
            
          
            if os.path.isdir(full_path): continue
            
            try:
        
                dcm = pydicom.dcmread(full_path, stop_before_pixels=True, force=True)
                modality = dcm.get("Modality", "Unknown")
                
                
                if modality == 'CT':
                  
                    new_path = os.path.join(ct_dir, f)
                    shutil.move(full_path, new_path)
                    files_moved += 1
                    
            except:
                continue 

    print(f"\n\nSUCCESS! All Ahsania patients are now reorganized.")
else:
    print("Source path not found.")

In [3]:
import os
import shutil
import pydicom

source_path = '../Data/SQUARE_F/'

print(f"--- STARTING SMART CLEANING (SQUARE) ---")

def get_referenced_uid(rtstruct_path):
    try:
        dcm = pydicom.dcmread(rtstruct_path, stop_before_pixels=True, force=True)
        if 'ReferencedFrameOfReferenceSequence' in dcm:
            rfor = dcm.ReferencedFrameOfReferenceSequence[0]
            if 'RTReferencedStudySequence' in rfor:
                rstudy = rfor.RTReferencedStudySequence[0]
                if 'RTReferencedSeriesSequence' in rstudy:
                    return rstudy.RTReferencedSeriesSequence[0].SeriesInstanceUID
    except: return None
    return None

if os.path.exists(source_path):
    patient_folders = sorted([f for f in os.listdir(source_path) if os.path.isdir(os.path.join(source_path, f))])
    print(f"Found {len(patient_folders)} patients.")
    
    for i, patient_id in enumerate(patient_folders):
        patient_dir = os.path.join(source_path, patient_id)
        
        rt_struct_path = None
        for root, dirs, files in os.walk(patient_dir):
            for f in files:
                if 'rs.' in f.lower() or (f.lower().startswith('rs') and '.dcm' in f.lower()):
                    rt_struct_path = os.path.join(root, f)
                    break
            if rt_struct_path: break
            
        if not rt_struct_path:
            print(f"[{i+1}] {patient_id}: [SKIP] No RTStruct found")
            continue

        target_uid = get_referenced_uid(rt_struct_path)
        if not target_uid:
            print(f"[{i+1}] {patient_id}: [SKIP] RTStruct has no UID")
            continue
            
        ct_dir = os.path.join(patient_dir, 'CT')
        struct_dir = os.path.join(patient_dir, 'Struct')
        
        if not os.path.exists(ct_dir): os.makedirs(ct_dir)
        if not os.path.exists(struct_dir): os.makedirs(struct_dir)
        
        moved_count = 0
        for root, dirs, files in os.walk(patient_dir):
            
            if 'CT' in root or 'Struct' in root: continue
            
            for f in files:
                full_path = os.path.join(root, f)
                try:
                    dcm = pydicom.dcmread(full_path, stop_before_pixels=True, force=True)
                    
                    if dcm.SeriesInstanceUID == target_uid:
                        shutil.move(full_path, os.path.join(ct_dir, f))
                        moved_count += 1
                        
                    elif full_path == rt_struct_path:
                        shutil.move(full_path, os.path.join(struct_dir, f))
                        
                except: continue
        
        print(f"[{i+1}] {patient_id}: Organized {moved_count} CT files.")

    print("\nSUCCESS! Square dataset cleaned.")
else:
    print("Source path not found.")

--- STARTING SMART CLEANING (SQUARE) ---
Found 121 patients.
[1] R130505087: Organized 0 CT files.
[2] R1406007291: Organized 0 CT files.
[3] R1508007367: Organized 0 CT files.
[4] R1701004331: Organized 0 CT files.
[5] R1702006414: Organized 0 CT files.
[6] R1707004823: Organized 0 CT files.
[7] R1710009613: Organized 0 CT files.
[8] R1803003706: Organized 0 CT files.
[9] R1807000334: Organized 0 CT files.
[10] R1807003460: Organized 0 CT files.
[11] R1807005275: Organized 0 CT files.
[12] R1807007064: Organized 0 CT files.
[13] R1808004062: Organized 0 CT files.
[14] R1810004752: Organized 0 CT files.
[15] R1810006592: Organized 0 CT files.
[16] R1810007735: Organized 0 CT files.
[17] R1811001641: Organized 0 CT files.
[18] R1903001183: Organized 0 CT files.
[19] R1903004587: Organized 0 CT files.
[20] R1905000571: Organized 0 CT files.
[21] R1905002341: Organized 0 CT files.
[22] R1905004532: Organized 0 CT files.
[23] R1907005733: Organized 0 CT files.
[24] R1907009307: Organized 0

In [2]:
import os
import shutil
import pydicom
from pydicom.errors import InvalidDicomError

# Configuration: Target cohort path
SOURCE_PATH = "../Data/SQUARE_F"  # Switch to AMCGH as needed

def extract_referenced_series_uid(rtstruct_path: str):
    """
    # Target Alignment: Queries DICOM RTSTRUCT metadata to extract the Referenced Series Instance UID,
    # ensuring spatial linkage between the 3D contour and the primary planning CT.
    """
    try:
        dcm = pydicom.dcmread(rtstruct_path, stop_before_pixels=True, force=True)
        if "ReferencedFrameOfReferenceSequence" in dcm:
            rfor = dcm.ReferencedFrameOfReferenceSequence[0]
            if "RTReferencedStudySequence" in rfor:
                rstudy = rfor.RTReferencedStudySequence[0]
                if "RTReferencedSeriesSequence" in rstudy:
                    return str(rstudy.RTReferencedSeriesSequence[0].SeriesInstanceUID)
    except Exception:
        return None
    return None

def harmonize_dicom_hierarchy(patient_dir: str):
    """
    # Universal Folder Organizer: Resolves mixed-modality directory structures by forcing header 
    # inspections, bypassing severed UIDs, and dynamically filtering non-axial acquisitions[cite: 5].
    """
    ct_dir = os.path.join(patient_dir, "CT")
    struct_dir = os.path.join(patient_dir, "Struct")
    os.makedirs(ct_dir, exist_ok=True)
    os.makedirs(struct_dir, exist_ok=True)

    counts = {"CT": 0, "RTSTRUCT": 0, "RTDOSE": 0, "RTPLAN": 0, "Skipped": 0}

    # Pass 1: Identify RTSTRUCT to resolve the topological planning series UID
    rtstruct_path = None
    for root, _, files in os.walk(patient_dir):
        if any(folder in root for folder in ["CT", "Struct"]): continue
        for f in files:
            file_path = os.path.join(root, f)
            try:
                hdr = pydicom.dcmread(file_path, stop_before_pixels=True, force=True)
                if hdr.get("Modality", "").strip().upper() == "RTSTRUCT":
                    rtstruct_path = file_path
                    break
            except (InvalidDicomError, Exception): continue
        if rtstruct_path: break

    target_ct_uid = extract_referenced_series_uid(rtstruct_path) if rtstruct_path else None

    # Pass 2: Deterministic sorting and collision-free relocation
    for root, _, files in os.walk(patient_dir):
        if any(folder in root for folder in ["CT", "Struct"]): continue

        for f in files:
            file_path = os.path.join(root, f)
            try:
                dcm = pydicom.dcmread(file_path, stop_before_pixels=True, force=True)
                modality = dcm.get("Modality", "").strip().upper()
                dest_dir = None

                if modality == "CT":
                    # Topological Validation: Suppresses topogram/scout series contamination
                    if target_ct_uid is None or dcm.SeriesInstanceUID == target_ct_uid:
                        dest_dir = ct_dir
                        counts["CT"] += 1
                    else:
                        counts["Skipped"] += 1
                        continue
                elif modality == "RTSTRUCT":
                    dest_dir = struct_dir
                    counts["RTSTRUCT"] += 1
                elif modality in ["RTDOSE", "RTPLAN"]:
                    # Preserve physical dose-grid matrices in root for TPS native coordinate parity[cite: 5]
                    counts[modality] += 1
                    continue
                else:
                    counts["Skipped"] += 1
                    continue

                if dest_dir:
                    base_name, ext = os.path.splitext(f)
                    dest_path = os.path.join(dest_dir, f)
                    
                    # Collision Handler: Prevents silent data overwrite
                    collision_idx = 1
                    while os.path.exists(dest_path):
                        dest_path = os.path.join(dest_dir, f"{base_name}_dup{collision_idx}{ext}")
                        collision_idx += 1

                    shutil.move(file_path, dest_path)

            except (InvalidDicomError, Exception):
                counts["Skipped"] += 1
                continue
    return counts

if os.path.exists(SOURCE_PATH):
    patients = sorted([p for p in os.listdir(SOURCE_PATH) if os.path.isdir(os.path.join(SOURCE_PATH, p))])
    print(f"Cohort identified: {len(patients)} subjects.")
    
    for idx, p_id in enumerate(patients):
        stats = harmonize_dicom_hierarchy(os.path.join(SOURCE_PATH, p_id))
        print(f"[{idx+1}/{len(patients)}] Harmonized {p_id}: {stats}")
else:
    print("Source path not found.")

Cohort identified: 121 subjects.
[1/121] Harmonized R130505087: {'CT': 0, 'RTSTRUCT': 0, 'RTDOSE': 1, 'RTPLAN': 1, 'Skipped': 0}
[2/121] Harmonized R1406007291: {'CT': 0, 'RTSTRUCT': 0, 'RTDOSE': 1, 'RTPLAN': 1, 'Skipped': 0}
[3/121] Harmonized R1508007367: {'CT': 0, 'RTSTRUCT': 0, 'RTDOSE': 1, 'RTPLAN': 1, 'Skipped': 0}
[4/121] Harmonized R1701004331: {'CT': 0, 'RTSTRUCT': 0, 'RTDOSE': 1, 'RTPLAN': 1, 'Skipped': 0}
[5/121] Harmonized R1702006414: {'CT': 0, 'RTSTRUCT': 0, 'RTDOSE': 1, 'RTPLAN': 1, 'Skipped': 0}
[6/121] Harmonized R1707004823: {'CT': 0, 'RTSTRUCT': 0, 'RTDOSE': 1, 'RTPLAN': 1, 'Skipped': 0}
[7/121] Harmonized R1710009613: {'CT': 0, 'RTSTRUCT': 0, 'RTDOSE': 1, 'RTPLAN': 1, 'Skipped': 0}
[8/121] Harmonized R1803003706: {'CT': 0, 'RTSTRUCT': 0, 'RTDOSE': 1, 'RTPLAN': 1, 'Skipped': 0}
[9/121] Harmonized R1807000334: {'CT': 0, 'RTSTRUCT': 0, 'RTDOSE': 1, 'RTPLAN': 1, 'Skipped': 0}
[10/121] Harmonized R1807003460: {'CT': 0, 'RTSTRUCT': 0, 'RTDOSE': 1, 'RTPLAN': 1, 'Skipped': 

In [4]:
import os
import shutil
import pydicom

# Target cohort path
SOURCE_PATH = '../Data/AMCGH/'

def organize_and_cleanup(patient_dir):
    """
    Universal Organizer: Sorts legacy commingled DICOMs into /CT and /Struct 
    while preserving RTDOSE and RTPLAN in the patient root. Dynamically deletes 
    legacy empty folders[cite: 1, 4].
    """
    ct_dir = os.path.join(patient_dir, 'CT')
    struct_dir = os.path.join(patient_dir, 'Struct')
    os.makedirs(ct_dir, exist_ok=True)
    os.makedirs(struct_dir, exist_ok=True)

    counts = {'CT': 0, 'RTSTRUCT': 0, 'RTDOSE': 0, 'RTPLAN': 0, 'Skipped': 0}

    # Pass 1: Deterministic sorting
    for root, _, files in os.walk(patient_dir):
        if 'CT' in root or 'Struct' in root: continue
        for f in files:
            file_path = os.path.join(root, f)
            try:
                dcm = pydicom.dcmread(file_path, stop_before_pixels=True, force=True)
                modality = dcm.get("Modality", "").strip().upper()
                dest_dir = None
                
                if modality == 'CT':
                    dest_dir = ct_dir
                    counts['CT'] += 1
                elif modality == 'RTSTRUCT':
                    dest_dir = struct_dir
                    counts['RTSTRUCT'] += 1
                elif modality in ['RTDOSE', 'RTPLAN']:
                    counts[modality] += 1
                    continue # Preserve natively in root
                else:
                    counts['Skipped'] += 1
                    continue

                if dest_dir:
                    base_name, ext = os.path.splitext(f)
                    dest_path = os.path.join(dest_dir, f)
                    collision_idx = 1
                    while os.path.exists(dest_path):
                        dest_path = os.path.join(dest_dir, f"{base_name}_dup{collision_idx}{ext}")
                        collision_idx += 1
                    shutil.move(file_path, dest_path)
            except Exception:
                counts['Skipped'] += 1
                continue

    # Pass 2: Garbage collection for artifact directories
    for garbage_dir in ['Dose', 'Plan']:
        garbage_path = os.path.join(patient_dir, garbage_dir)
        if os.path.exists(garbage_path) and not os.listdir(garbage_path):
            os.rmdir(garbage_path)

    return counts

if os.path.exists(SOURCE_PATH):
    patients = sorted([p for p in os.listdir(SOURCE_PATH) if os.path.isdir(os.path.join(SOURCE_PATH, p))])
    print("--- STARTING UNIVERSAL ORGANIZER (CLEANUP MODE) ---")
    for idx, p_id in enumerate(patients):
        stats = organize_and_cleanup(os.path.join(SOURCE_PATH, p_id))
        print(f"[{idx+1}/{len(patients)}] {p_id} | CT: {stats['CT']}, Struct: {stats['RTSTRUCT']} | Dose/Plan kept in root.")
else:
    print("Source path not found.")

--- STARTING UNIVERSAL ORGANIZER (CLEANUP MODE) ---
[1/29] 20250982new | CT: 0, Struct: 0 | Dose/Plan kept in root.
[2/29] 20251575new | CT: 0, Struct: 0 | Dose/Plan kept in root.
[3/29] 20251577new | CT: 0, Struct: 0 | Dose/Plan kept in root.
[4/29] 20251626new | CT: 0, Struct: 0 | Dose/Plan kept in root.
[5/29] 20251673new | CT: 0, Struct: 0 | Dose/Plan kept in root.
[6/29] 20251705new | CT: 0, Struct: 0 | Dose/Plan kept in root.
[7/29] 20251727new | CT: 0, Struct: 0 | Dose/Plan kept in root.
[8/29] 20251773 | CT: 0, Struct: 0 | Dose/Plan kept in root.
[9/29] 20260104 | CT: 0, Struct: 0 | Dose/Plan kept in root.
[10/29] 20260104 - Copy | CT: 0, Struct: 0 | Dose/Plan kept in root.
[11/29] New 20250871 | CT: 0, Struct: 0 | Dose/Plan kept in root.
[12/29] New 20251222 | CT: 0, Struct: 0 | Dose/Plan kept in root.
[13/29] New 20251320 | CT: 0, Struct: 0 | Dose/Plan kept in root.
[14/29] New 20251360 | CT: 0, Struct: 0 | Dose/Plan kept in root.
[15/29] New 20251384 | CT: 0, Struct: 0 | Dos